## Interpolation theorem in Signal processing

A classic result in signal processing is the **interpolation theorem**, stating that it is possible to exactly recover a whole signal $(x(t))_{t \in \mathbb{R}}$ over all $\mathbb{R}$ by only knowing its value on the discrete grid $(nT_e)_{n \in \mathbb{Z}}$, at the sole condition that $T_e$, the sampling period, verifyies the condition $F_e=\frac1{T_e} \geq 2B$, where $B=f_{\text{max}}-f_{\text{min}}$. $f_{\text{max}}$ and $f_{\text{min}} \geq 0$ are respectively the largest and smallest frequencies in the spectre of $x$. Of course, if $f_{\text{max}} = +\infty$, then this theorem does not provide an answer, but it might be impossible. Also, notice that if $f_{\text{min}}=0$, then one recovers the well-known Shannon's theorem, stating that the condition to prevent anti-aliasing is $F_e \geq 2f_{\text{max}}$.

Moreover, the theorem also provides a reconstruction method, known as **Whittaker-Shannon** interpolation formula:

$\forall t \in \mathbb{R}$:
$$
x(t) = \sum_{n=-\infty}^{+\infty} x(nT_e)\frac{\sin(\pi F_e(t-nT_e))}{\pi F_e(t-nT_e)} = \sum_{n=-\infty}^{+\infty} x(nT_e)\text{sinc}(\pi F_e(t-nT_e)).
$$

I am going to check this formula by implementing Python code and try to gain an intuition on its speed of convergence. The computer can not store an infinite sequence, or compute an infinite serie, so we are going to truncate it via an integer $N$, that is:
$$
\tilde x(t)= \sum_{n=-N}^{N} x(nT_e)\frac{\sin(\pi F_e(t-nT_e))}{\pi F_e(t-nT_e)}
$$
and observe how well $\tilde x(t)$ approximates $x(t)$ by comparing graphs.

Assume that one wants to plot $\hat x$ over an interval $[T_a, T_b]$. To use $x$ over this whole interval, one must choose $N \geq \displaystyle\lceil \frac{T_b-T_a}{2T_e} \rceil$.

**In practice however, this method is not popular for signal reconstruction.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def exact_interpolation(f: callable, T: np.ndarray, N: int, Fe: float):
    """Compute an approximation of the exact interpolation formula, see above.
    The total number of element in the sum is 2N+1.

    Args:
        f (callable): signal to approximate.
        T (np.ndarray): Time points where to evalute interpolation.
        N (int): Number of signal element on time positive axis.
        X (np.ndarray): In case f is not provided, X should be the signal recorded at sampling frequency Fe.
    """
    Te=1/Fe
    Int=np.linspace(-N, N, 2*N+1)*Te
    return np.sum(f(Int)*np.sinc(Fe*(T[:, None]-Int)), axis=-1)

def compare_with_plots(f:callable, T:np.ndarray, X:np.ndarray=None, ax=None):
    if ax is None:
        ax=plt.gca()

    Fe=1e5
    N_min=np.ceil((T[-1] - T[0])*Fe/2).astype(int)
    N=[np.floor(N_min*.5).astype(int), N_min]
    Result=[]
    F=f(T)
    for n in N:
        Result += [exact_interpolation(f=f, T=T, N=n, Fe=Fe)]
    ax.plot(T,F, label='Exact function')
    for i,n in enumerate(N):
        ax.plot(T, Result[i], label=f"N={n}")
    ax.set_title(f'Plots of true function and its truncated interpolations, with Fe={int(Fe)}.')
    plt.legend()

fig=plt.figure(figsize=(15,10))

#Very regular function
sin_cos=lambda x: np.sin(10*x) + np.cos(x)

ax=fig.add_subplot(211)

compare_with_plots(sin_cos, T=np.linspace(-10,10,201), ax=ax)

#Very regular function
sin_cos=lambda x: np.sin(x) + np.cos(x)

ax=fig.add_subplot(212)

compare_with_plots(sin_cos, T=np.linspace(-10,10,201), ax=ax)
plt.show()